# CITADEL Single-Notebook Experiment Runner

This notebook is the one-stop runner for the CITADEL journal-extension workspace. It locates the tracked telemetry data, reproduces the EXACT baseline, runs the CITADEL/TCAD ablation grid, attaches hardware-cost estimates, exports FPGA/RTL golden vectors, and shows how future RTL synthesis results can be merged back into the paper tables.

The default settings use the tracked DDR telemetry in `data/telemetry/processed/ddr_data/` with the smoke grid. For final TCAD numbers, keep `DATA_MODE = "real"` and set `TCAD_PRESET` to `"full"`. Apple tier data is located at `data/telemetry/raw/apple_data/` and is kept separate from CINTAS hardware-cost claims.

In [ ]:
from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "exact").is_dir():
            return candidate
    raise RuntimeError(f"Could not find CITADEL repo root from {start}")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository: {REPO_ROOT}")
print(f"Python: {platform.python_version()} on {platform.platform()}")

## 1. Reproducibility Configuration

The notebook uses deterministic settings for every experiment:

- seed: `123`
- numerical threads: `1`
- automatic DDR and Apple telemetry discovery
- deterministic sample-data generator when `DATA_MODE = "sample"`
- repo-relative paths
- run manifests with package versions, git commit, input hashes, and output hashes

For the strictest cross-machine comparison, start Jupyter itself with `PYTHONHASHSEED=123`. The notebook still sets the environment variable for downstream tools.

In [ ]:
SEED = 123
THREADS = 1
SAMPLE_ROWS = 600

DDR_DATA_ROOT = REPO_ROOT / "data" / "telemetry" / "processed" / "ddr_data"
APPLE_DATA_ROOT = REPO_ROOT / "data" / "telemetry" / "raw" / "apple_data"
REAL_DATA_ROOT = DDR_DATA_ROOT
APPLE_TIER_DATA_ROOT = APPLE_DATA_ROOT
DATA_SOURCE_CONFIG = REPO_ROOT / "data" / "external_sources.json"

def csv_files(root: Path) -> list[Path]:
    return sorted(root.rglob("*.csv"))


def is_git_lfs_pointer(path: Path) -> bool:
    try:
        with path.open("rb") as f:
            return f.read(64).startswith(b"version https://git-lfs.github.com/spec")
    except OSError:
        return False


DDR_CSVS = csv_files(DDR_DATA_ROOT)
APPLE_CSVS = csv_files(APPLE_DATA_ROOT)
pointer_files = [p for p in [*DDR_CSVS[:3], *APPLE_CSVS[:3]] if is_git_lfs_pointer(p)]
if pointer_files:
    listed = "\n".join(str(p.relative_to(REPO_ROOT)) for p in pointer_files[:6])
    raise RuntimeError(
        "Telemetry CSVs are still Git LFS pointer files. Fetch the Git LFS objects "
        "with your Git client before running the notebook. Examples:\n" + listed
    )

# The repo now tracks real telemetry. Change to "sample" only for a tiny pipeline check.
DATA_MODE = "real" if DDR_CSVS else "sample"  # "real" or "sample"
TCAD_PRESET = "smoke"  # "smoke" for laptop/debug, "full" for journal-scale sweeps
RUN_REPEAT_CHECK = True

RESULTS_ROOT = REPO_ROOT / "results" / "notebook_run"
DATA_ROOT = REPO_ROOT / "data" / "sample" if DATA_MODE == "sample" else REAL_DATA_ROOT
ETS_OUT = RESULTS_ROOT / "ets_baseline"
TCAD_OUT = RESULTS_ROOT / "tcad_ablation"
TCAD_REPEAT_OUT = RESULTS_ROOT / "tcad_ablation_repeat"
FPGA_OUT = RESULTS_ROOT / "fpga"
RTL_SWEEP_OUT = RESULTS_ROOT / "rtl_sweep"

from exact.repro import configure_reproducibility

env_updates = configure_reproducibility(seed=SEED, threads=THREADS, matplotlib_backend="Agg")
for key, value in env_updates.items():
    print(f"{key}={value}")

## 2. Prepare Data

This cell verifies the tracked telemetry folders used by the notebook. DDR4/DDR5 data is read from `data/telemetry/processed/ddr_data/`. Apple tier-0/1/2 data is read from `data/telemetry/raw/apple_data/` for limited-observability studies; it is not mixed into CINTAS hardware-cost claims. If `DATA_MODE` is set to `sample`, the notebook generates deterministic synthetic telemetry for a tiny pipeline check.

In [ ]:
from exact.sample_data import create_sample_dataset

if DATA_SOURCE_CONFIG.exists():
    sources = json.loads(DATA_SOURCE_CONFIG.read_text())
    display(pd.DataFrame([
        {"source": key, "repo": val["repo"], "target": val["target"], "kind": val.get("kind", "")}
        for key, val in sources.items()
    ]))

display(pd.DataFrame([
    {
        "dataset": "ddr_data",
        "root": str(DDR_DATA_ROOT.relative_to(REPO_ROOT)),
        "csv_files": len(DDR_CSVS),
        "role": "CITADEL CINTAS experiments",
    },
    {
        "dataset": "apple_data",
        "root": str(APPLE_DATA_ROOT.relative_to(REPO_ROOT)),
        "csv_files": len(APPLE_CSVS),
        "role": "limited-observability analysis",
    },
]))

if DATA_MODE == "sample":
    created = create_sample_dataset(DATA_ROOT, seed=SEED, n_rows=SAMPLE_ROWS)
    print(f"Generated {len(created)} deterministic telemetry CSVs under {DATA_ROOT}")
else:
    csvs = sorted(DATA_ROOT.glob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSV files found in {DATA_ROOT}")
    expected = 78
    if len(csvs) != expected:
        print(f"Warning: expected {expected} DDR CSVs, found {len(csvs)}")
    print(f"Using {len(csvs)} DDR telemetry CSVs from {DATA_ROOT}")

sample_files = sorted(DATA_ROOT.glob("*.csv"))[:8]
display(pd.DataFrame({"example_input_files": [str(p.relative_to(REPO_ROOT)) for p in sample_files]}))

## 3. ETS-Style Baseline Reproduction

This section runs the conference-version pipeline on the selected telemetry snapshot:

1. load Setup A and Setup B telemetry
2. clean and debias telemetry
3. fit benign-only CINTAS calibration
4. build causal/ranking artifacts
5. evaluate decision-block metrics
6. write figures, CSV summaries, and a run manifest

In [ ]:
from exact.experiments.ets2026 import ETS2026Config, run_ets2026

ets_cfg = ETS2026Config(
    window_sizes=(50, 100, 200),
    n_splits=3,
    lambda_res=0.5,
    agg_mode="max",
    seed=SEED,
)

ets_artifacts = run_ets2026(data_root=DATA_ROOT, out_root=ETS_OUT, cfg=ets_cfg)
print(f"ETS manifest: {ets_artifacts['run_manifest'].relative_to(REPO_ROOT)}")
print(f"Shared features: {len(ets_artifacts['shared_features'])}")

ets_a = pd.read_csv(ETS_OUT / "SETUP_A_EXACT_summary_global.csv")
ets_b = pd.read_csv(ETS_OUT / "SETUP_B_EXACT_summary_global.csv")
display(pd.concat([ets_a.assign(setup="A"), ets_b.assign(setup="B")], ignore_index=True).head(12))

## 4. TCAD Design-Space Ablation

This section runs the TCAD extension sweep. The smoke preset is intentionally small; the full preset expands feature budgets, decision-block sizes, score weights, aggregation choices, and fixed-point precisions.

Each summary row includes detection metrics and hardware-cost columns derived from the operator table under `hardware/`.

In [ ]:
from exact.experiments.tcad2026 import TCAD2026Config, run_tcad_ablation

cfg_path = REPO_ROOT / "configs" / f"tcad_grid_{TCAD_PRESET}.json"
tcad_cfg = TCAD2026Config.from_json(cfg_path)
print(f"TCAD config: {cfg_path.relative_to(REPO_ROOT)}")

# Keep the notebook seed explicit even when the config file changes.
tcad_cfg = TCAD2026Config(
    scenarios_eval=tcad_cfg.scenarios_eval,
    feature_budgets=tcad_cfg.feature_budgets,
    window_sizes=tcad_cfg.window_sizes,
    lambda_res_values=tcad_cfg.lambda_res_values,
    agg_modes=tcad_cfg.agg_modes,
    weight_modes=tcad_cfg.weight_modes,
    fixed_point_q=tcad_cfg.fixed_point_q,
    n_splits=tcad_cfg.n_splits,
    p_quantile=tcad_cfg.p_quantile,
    corr_threshold=tcad_cfg.corr_threshold,
    seed=SEED,
)

tcad_artifacts = run_tcad_ablation(data_root=DATA_ROOT, out_root=TCAD_OUT, cfg=tcad_cfg)
summary = tcad_artifacts["summary"].copy()
print(f"TCAD summary: {tcad_artifacts['summary_path'].relative_to(REPO_ROOT)}")
print(f"TCAD manifest: {tcad_artifacts['manifest_path'].relative_to(REPO_ROOT)}")
display(summary.head(10))

## 5. Reproducibility Check

This cell reruns the TCAD ablation into a second output directory and checks that the summary table is bit-for-bit identical at the DataFrame level. This is the notebook equivalent of the repo smoke test.

In [ ]:
if RUN_REPEAT_CHECK:
    repeat_artifacts = run_tcad_ablation(data_root=DATA_ROOT, out_root=TCAD_REPEAT_OUT, cfg=tcad_cfg)
    repeat_summary = repeat_artifacts["summary"].copy()
    same = summary.equals(repeat_summary)
    print(f"Repeated TCAD summary equals first run: {same}")
    if not same:
        diff_cols = [c for c in summary.columns if not summary[c].equals(repeat_summary[c])]
        raise AssertionError(f"Reproducibility check failed; differing columns: {diff_cols}")
else:
    print("RUN_REPEAT_CHECK=False, skipped repeat execution.")

## 6. Manifest And Artifact Audit

Run manifests are the reproducibility contract. They capture the config, runtime environment, package versions, input hashes, output hashes, and git commit.

In [ ]:
def load_manifest(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

ets_manifest = load_manifest(ETS_OUT / "run_manifest.json")
tcad_manifest = load_manifest(TCAD_OUT / "run_manifest.json")

manifest_overview = pd.DataFrame([
    {
        "run": "ETS baseline",
        "git_commit": ets_manifest.get("git_commit"),
        "data_files": len(ets_manifest.get("data_files", [])),
        "artifacts": len(ets_manifest.get("artifacts", [])),
        "python": ets_manifest.get("python_version"),
    },
    {
        "run": "TCAD ablation",
        "git_commit": tcad_manifest.get("git_commit"),
        "data_files": len(tcad_manifest.get("data_files", [])),
        "artifacts": len(tcad_manifest.get("artifacts", [])),
        "python": tcad_manifest.get("python_version"),
    },
])
display(manifest_overview)

display(pd.DataFrame(tcad_manifest.get("data_files", [])).head())

## 7. Hardware-Cost Summary

This section uses the hardware information you provided from Eduardo Ortega's script.

Operator source table:

- add: raw area `1165.234`, power `0.178 mW`, delay `62.7 ps`, cycles `3`
- mult: raw area `4532.164`, power `0.5146 mW`, delay `29.09 ps`, cycles `2`
- raw area is divided by `1000**2`, matching the reference script
- STD block per feature: `2 * mult + add`
- STD adder tree: `(n_features - 1) * add`
- AGG block: `2 * mult`
- power scales linearly with GHz

In [ ]:
from exact.hardware import OperatorCosts, compute_tableIII_setupB, estimate_cintas_hardware_cost

costs = OperatorCosts.from_csv(REPO_ROOT / "hardware" / "cintas_operator_costs.csv")
operator_table = pd.DataFrame([
    {"operator": "add", "area_mm2": costs.add_area_mm2, "power_mw_at_1ghz": costs.add_power_mw_at_1ghz, "delay_ps": costs.add_delay_ps, "cycles": costs.add_cycles},
    {"operator": "mult", "area_mm2": costs.mult_area_mm2, "power_mw_at_1ghz": costs.mult_power_mw_at_1ghz, "delay_ps": costs.mult_delay_ps, "cycles": costs.mult_cycles},
])
display(operator_table)

display(compute_tableIII_setupB(n_features=15, frequency_ghz=1.0))

hardware_cols = [
    "setup", "scenario", "top_k", "n_selected_features", "fixed_point_q",
    "hw_area_mm2", "hw_power_mw", "hw_setup_b_area_overhead_pct",
    "hw_idle_power_overhead_pct", "hw_add_count", "hw_mult_count",
]
display(summary[hardware_cols].drop_duplicates().head(12))

## 8. FPGA/RTL Laptop Workflow

You can approach FPGA work on a laptop in three layers.

### Layer A: No FPGA board required

1. Use Python to export fixed-point golden vectors.
2. Simulate RTL against those vectors.
3. Debug bit-exact arithmetic and stream timing.
4. Save simulation pass/fail summaries under `results/rtl_sweep/`.

This is enough to connect RTL correctness to the notebook.

### Layer B: Open-source synthesis on laptop

1. Install a simulator such as Verilator or Icarus Verilog.
2. Install Yosys for synthesis experiments.
3. For small open FPGA targets, add the matching place-and-route tools.
4. Run synthesis through notebook-managed cells or import the completed tool reports.
5. Write `rtl_resource_summary.csv` with LUT/FF/DSP/BRAM, timing, cycles, and estimated energy.

This gives preliminary hardware evidence, but it depends on the target family.

### Layer C: Vendor FPGA flow

1. Pick a target board and FPGA family.
2. Use the vendor toolchain for authoritative synthesis and implementation reports.
3. On macOS laptops, vendor FPGA tools are often not native; a Linux workstation, server, or VM is usually the practical path.
4. Export the reports to CSV.
5. Let this notebook merge the CSV with the TCAD ablation table.

The notebook integration path is: Python fixed-point model -> golden vectors -> RTL simulation/synthesis -> CSV report -> merged TCAD paper table.

In [ ]:
tools = []
for name, command in [
    ("verilator", ["verilator", "--version"]),
    ("iverilog", ["iverilog", "-V"]),
    ("yosys", ["yosys", "-V"]),
    ("gtkwave", ["gtkwave", "--version"]),
]:
    exe = shutil.which(command[0])
    row = {"tool": name, "available": exe is not None, "path": exe or ""}
    if exe:
        try:
            completed = subprocess.run(command, capture_output=True, text=True, timeout=10)
            first_line = (completed.stdout or completed.stderr).splitlines()[0] if (completed.stdout or completed.stderr) else ""
            row["version"] = first_line
        except Exception as exc:
            row["version"] = f"version check failed: {exc}"
    else:
        row["version"] = "not installed"
    tools.append(row)

display(pd.DataFrame(tools))

## 9. Export Fixed-Point Golden Vectors For RTL

This cell exports a small golden-vector file from the Python fixed-point CINTAS reference. The RTL testbench should stream these feature values and compare its output score against `expected_score_q`.

In [ ]:
from exact.cintas import FixedPointCINTAS, FixedPointConfig

FPGA_OUT.mkdir(parents=True, exist_ok=True)
model = ets_artifacts["model_A"]
setup_a = ets_artifacts["setup_A"]
q_format = 15
fixed = FixedPointCINTAS.from_float_model(model, FixedPointConfig(q=q_format))
subset = setup_a[model.feature_cols].head(32).copy()
score_float, e1_float, e2_float = model.score_dataframe(subset)
score_q = fixed.score_dataframe(subset)

golden = subset.copy()
golden.insert(0, "sample_index", range(len(golden)))
golden["expected_score_q"] = score_q
golden["expected_score_float"] = score_float
golden["expected_e1_float"] = e1_float
golden["expected_e2_float"] = e2_float
golden_path = FPGA_OUT / f"cintas_setupA_q{q_format}_golden_vectors.csv"
golden.to_csv(golden_path, index=False)

print(f"Golden vectors: {golden_path.relative_to(REPO_ROOT)}")
display(golden.head())

## 10. Merge Future RTL/FPGA Results Into The TCAD Table

When simulation or synthesis is ready, save a CSV at `results/notebook_run/rtl_sweep/rtl_resource_summary.csv` or `results/rtl_sweep/rtl_resource_summary.csv` with columns like:

```text
setup,top_k,fixed_point_q,luts,ffs,dsps,brams,fmax_mhz,latency_cycles,energy_per_block_nj
```

The cell below loads that file if it exists. Otherwise it writes a template so the expected schema is clear.

In [ ]:
rtl_summary_candidates = [
    RTL_SWEEP_OUT / "rtl_resource_summary.csv",
    REPO_ROOT / "results" / "rtl_sweep" / "rtl_resource_summary.csv",
]
existing = next((p for p in rtl_summary_candidates if p.exists()), None)

if existing is None:
    RTL_SWEEP_OUT.mkdir(parents=True, exist_ok=True)
    template = pd.DataFrame([
        {
            "setup": "A",
            "top_k": int(summary["top_k"].iloc[0]),
            "fixed_point_q": int(summary["fixed_point_q"].iloc[0]),
            "luts": None,
            "ffs": None,
            "dsps": None,
            "brams": None,
            "fmax_mhz": None,
            "latency_cycles": None,
            "energy_per_block_nj": None,
            "status": "fill after RTL simulation/synthesis",
        }
    ])
    existing = RTL_SWEEP_OUT / "rtl_resource_summary.csv"
    template.to_csv(existing, index=False)
    print(f"Created RTL summary template: {existing.relative_to(REPO_ROOT)}")

rtl_summary = pd.read_csv(existing)
print(f"RTL summary source: {existing.relative_to(REPO_ROOT)}")
display(rtl_summary.head())

merge_keys = [key for key in ["setup", "top_k", "fixed_point_q"] if key in rtl_summary.columns and key in summary.columns]
if merge_keys:
    merged = summary.merge(rtl_summary, on=merge_keys, how="left")
    display(merged.head())
else:
    print("RTL summary does not yet share merge keys with the TCAD summary.")

## 11. Paper-Ready Output Checklist

After this notebook completes, use these generated artifacts to keep the journal work organized:

- ETS baseline manifest: `results/notebook_run/ets_baseline/run_manifest.json`
- TCAD ablation manifest: `results/notebook_run/tcad_ablation/run_manifest.json`
- TCAD ablation summary: `results/notebook_run/tcad_ablation/tcad_ablation_summary.csv`
- selected features and COM/MEM/SEN counts: `results/notebook_run/tcad_ablation/tcad_selected_features.csv`
- fixed-point golden vectors: `results/notebook_run/fpga/cintas_setupA_q15_golden_vectors.csv`
- RTL/FPGA result template or merged report: `results/notebook_run/rtl_sweep/rtl_resource_summary.csv`

For the final journal run, switch from deterministic sample data to the frozen real telemetry snapshot and run the full TCAD grid.

In [ ]:
final_artifacts = [
    ETS_OUT / "run_manifest.json",
    TCAD_OUT / "run_manifest.json",
    TCAD_OUT / "tcad_ablation_summary.csv",
    TCAD_OUT / "tcad_selected_features.csv",
    golden_path,
    existing,
]

for artifact in final_artifacts:
    print(f"{'OK' if artifact.exists() else 'MISSING'}  {artifact.relative_to(REPO_ROOT)}")

print("\nNotebook run complete.")